# Lab 4 - Power Analysis to Bypass Password

---
NOTE: This lab references some (commercial) training material on [ChipWhisperer.io](https://learn.ChipWhisperer.io/courses/sca101). You can freely execute and use the lab per the open-source license (including using it in your own courses if you distribute similarly), but you must maintain notice about this source location. Consider joining our training course to enjoy the full experience.

---

**SUMMARY:** *This tutorial will introduce you to breaking devices by determining when a device is performing certain operations. Our target device will be performing a password check, and we will demonstrate how to perform simple power analysis.*

**LEARNING OUTCOMES:**

* How power can be used to determine timing information.
* Plotting multiple iterations while varying input data to find interesting locations.
* Using difference of waveforms to find interesting locations.


## Prerequisites

Hold up! Before you continue, check you've done the following tutorial:

* ☑ [Lab 3 - Power Analysis and Code Flow](../Lab%203%20-%20Power%20Analysis%20and%20Code%20Flow/Lab.ipynb).

## Power Trace Gathering

## Setup

Let's start off by connecting to our ChipWhisperer-Husky and running the default setup, just like in the last lab.

## Target Firmware

As we saw in the previous lab, we can use power traces to gain some idea of what the target is doing - for example, a loop versus multiple repeated instructions, or expensive versus simpler instructions. The goal for this lab is to put this knowledge to practical use by breaking a simple password check.

For this lab, we're going to be using some different firmware. Instead of the `simpleserial-base` project, we're going to be using the `basic-passwdcheck` firmware. This firmware does the following: it boots, prints some stuff, waits for a password over serial, compares the password to one in the firmware, then sends a serial message back if the password is correct. If it's incorrect, it informs us that our password was incorrect, then goes into an infinite loop. Let's build this firmware and program our SAM4S:

In [ ]:
%%bash
cd ../../firmware/mcu/basic-passwdcheck
make PLATFORM=CWHUSKY CRYPTO_TARGET=NONE SS_VER=SS_VER_1_1 -j

In [ ]:
cw.program_target(scope, cw.programmers.SAM4SProgrammer, "../../firmware/mcu/basic-passwdcheck/basic-passwdcheck-CWHUSKY.hex")

Now that we've got our SAM4S programmed, let's try talking to it. Unlike the previous lab, this firmware doesn't use SimpleSerial. Instead, we're just going to be sending raw ASCII characters terminated by a newline.

There's a few things to keep in mind here:

1. Because the target goes into an infinite loop after receiving an incorrect password, we should reset the target before each time we talk to it. You can do this by toggling `scope.io.nrst` between `0` (pulled low) and `None` (high-z) with a small delay (0.5 seconds should be safe).
2. The password is comprised entirely of numbers and lower case characters.
3. The target ignores all null bytes (0x00).
4. The password is up to 32 characters long
5. The response when you have a correct password is "Access granted, Welcome!"

We recommend creating the following function: `cap_pass_trace()`, that resets the target, arms the scope, sends a password guess, then captures a power trace and reads the response. It should take a password guess as an argument and return a power trace and the response from the target.

Feel free to construct additional functions to make this easier. For example, you may find it helpful to make a function that resets your SAM4S.

Some additional things you may want to do:

1. Reduce the number of samples from 5000 down to something like 1000 or 2500. The password check is pretty quick, so the plot will look a bit cleaner if we reduce the number of samples
2. Clear out the serial read buffer with `target.read()`. Remember that the target will print things over serial upon booting, so you may get warnings about dropped messages if you don't clear this buffer by reading it.
3. Add an optional argument to `cap_pass_trace()` that prints out the response to the password.

Now, capture a trace with a password guess and plot it:

## Exploration

So what can we do with this? While first off - I'm going to cheat, and tell you that we have a preset password that starts with `h`. But that's the only hint so far - what can you do? Let's try plotting a comparison of `h` to something else.

As a reminder, you can plot multiple things on the same graph by multiplying the plots together.

<img src="img/h_vs_0_plot.png" alt="SPA of Power Analysis h vs 0" width="850"/>

Hopefully you see a noticeable difference here. Feel free to plot a few other chracter guesses, but what this plot is telling us is
that the execution path depends on whether or not this first character is correct

OK interesting -- but what if we didn't know the first character of the password? Often, a good strategy is to compare a bunch of different inputs to see if there's a difference. In our case, that would be lower case letters and numbers.

Our password implementation only recognizes characters in the list `abcdefghijklmnopqrstuvwxyz0123456789`, so we're going to limit it to those valid characters for now.

Write some code in the following block that implements the following algorithm:

```python
plot = cw.plot([])
for CHARACTER in LIST_OF_VALID_CHARACTERS:
    trace = cap_pass_trace(CHARACTER + "\n")
    plot *= cw.plot(trace).opts(color=hv.Palette('Spectral'))

plot
```
        
The above isn't quite valid code - so massage it into place! Depending on how you set `scope.adc.samples`, your power traces may be very long - you might want to make a more narrow plot that only does the first say 500 samples of the power trace. You can do that by slicing the power trace: `trace[start:end]`. You may want to apply a different plot palette as well. Check out the [Jupyter Cheat Sheet notebook](../Jupyter%20Cheat%20Sheet.ipynb) to see how to do that.

The end result should be if you zoom in, you'll see there is a location where a single "outlier" trace doesn't follow the path of all the other traces. That is great news, since it means we learn something about the system from power analysis.

<img src="img/spa_password_list_char1.png" alt="SPA of Power Analysis against all inputs" width="850"/>

Using your loop - you can also try modifying the analysis to capture a correct "first" character, and then every other wrong second character. Do you see a difference you might be able to detect?

The pseudo-code would look something like this:

```python
plot = cw.plot([])
for CHARACTER in LIST_OF_VALID_CHARACTERS:
    trace = cap_pass_trace("h" + CHARACTER + "\n")
    plot *= cw.plot(trace)
```

Give that a shot in your earlier code-block, and then let's try and automate this attack to understand the data a little better.

## Automating an Attack against One Character

To start with - we're going to automate an attack against a **single** character of the password. Since we don't know the password (let's assume), we'll use a strategy of comparing all possible inputs together.

An easy way to do this might be to use something that we know can't be part of the valid password. As long as it's processed the same way, this will work just fine. So for now, let's use a password of `0x01`. We can compare the this byte to processing something else:

You should see that these invalid passwords seem to be processed the same as an incorrect password. This simplifies things a bit, as we don't need to write additional code accounting for the case where we guess the character correctly the first time.

Let's make this a little more obvious, and plot the difference between a known reference & every other capture. You need to write some code that does something like this:

```python
ref_trace = cap_pass_trace( "\x01\n")

for CHARACTER in LIST_OF_VALID_CHARACTERS:
    trace = cap_pass_trace(CHARACTER + "\n")
    plot(trace - ref_trace)
```

Again, you'll need to modify this a bit. You may want to use your earlier code as a starting code.

OK great - hopefully you now see one major "difference". It should look something like this:
    
<img src="img/spa_password_diffexample.png" alt="SPA with Difference" width="850"/>
    

What do do now? Let's make this thing automatically detect such a large difference. Some handy stuff to try out is the `np.sum()` and `np.abs()` function.

The first one will get absolute values:

```python
import numpy as np
np.abs([-1, -3, 1, -5, 6])

    Out[]: array([1, 3, 1, 5, 6])
```

The second one will add up all the numbers.

```python
import numpy as np    
np.sum([-1, -3, 1, -5, 6])

    Out[]: -2
```

Using just `np.sum()` means positive and negative differences will cancel each other out - so it's better to do something like `np.sum(np.abs(DIFF))` to get a good number indicating how "close" the match was.


In [ ]:
import numpy as np
np.abs([-1, -3, 1, -5, 6])

In [ ]:
import numpy as np
np.sum([-1, -3, 1, -5, 6])

In [ ]:
np.sum(np.abs([-1, -3, 1, -5, 6]))

Taking your above loop, modify it to print an indicator of how closely this matches your trace. Something like the following should work:

```python
ref_trace = cap_pass_trace("\x01\n")

for CHARACTER in LIST_OF_VALID_CHARACTERS:
    trace = cap_pass_trace(CHARACTER + "\n")
    diff = SUM(ABS(trace - ref_trace))

    print("{:1} diff = {:2}".format(CHARACTER, diff))
```

Now the easy part - modify your above code to automatically print the correct password character. This should be done with a comparison of the `diff` variable - based on the printed characters, you should see one that is 'higher' than the others. 

There's a few different ways you can go about determining the most likely answer. You could try all the characters and  pick the highest. You could also simply pick a threshold and, if the difference is higher, select that as the correct guess.

The former is a bit more robust, but the latter will be faster.

## Running a Full Attack

Finally - let's finish this off. Rather than attacking a single character, we need to attack each character in sequence.

If you go back to the plotting of differences, you can try using the correct first character & wrong second character. The basic idea is exactly the same as before, but now we loop through up to 32 times, and just build up the password based on brute-forcing each character.

Take a look at the following for the basic pseudo-code:

```python
guessed_pw = "" #Store guessed password so far
pw_correct = False

do a loop 32 times (max password size):

    ref_trace = capture power trace(guessed_pw + "\x01\n")

    for CHARACTER in LIST_OF_VALID_CHARACTERS:
        trace, resp = capture power trace (guessed_pw + CHARACTER + newline)
        diff = SUM(ABS(trace - ref_trace))

        if diff > biggest_diff:
            best_guess = CHARACTER
            biggest_diff = diff

        if resp is BROKEN:
            pw_correct = True
            break
            
    if pw_correct:
        break
            
```

You should get an output that looks like this:

```
    h
    h0
    h0p
    h0px
    h0px3
    Access granted, Welcome!
```

If so - 🥳🥳🥳🥳🥳🥳🥳🥳🥳🥳🥳🥳🥳 Congrats - you did it!!!!

If not - check some troubleshooting hints below.

## Troubleshooting - Always get 'h'

Some common problems you might run into - first, if you get an output which keeps guessing the first character:

```
h
hh
hhh
hhhh
hhhhh
```

Check that when you run the `cap_pass_trace` inside the loop (checking the guessed password), are you updating the prefix of the password? For example, the old version of the code (guessing a single character) looked like this:

```python
trace = cap_pass_trace(c + "\n")[0]
```

But that is always sending our first character only! So we need to send the "known good password so far". In the example code something like this:

```
trace = cap_pass_trace(guessed_pw + c + "\n")[0]
```

Where `guessed_pw` progressively grows with the known good start of the password.

## Troubleshooting - Always get 'a'

This looks like it's always matching the first character:

```
h
ha
haa
haaa
haaaa
```

Check that you update the `ref_trace` - if you re-use the original reference trace, you won't be looking at a reference where the first N characters are good, and the remaining characters are bad. An easy way to do this is again using the `guessed_pw` variable and appending an invalid character + newline:

```python
trace = cap_pass_trace(guessed_pw + "\x01\n")[0]
```

## Troubleshooting - Can't break final characters

If your attack is unreliable towards the end, you may have set `scope.adc.samples` too low. Shorter traces have less of a difference than longer traces due to the desynchronization of the correct guess from the reference trace. Setting this
too low may also be cutting off the check entirely!

In [ ]:
scope.dis()
target.dis()

---
<small>NO-FUN DISCLAIMER: This material is Copyright (C) NewAE Technology Inc., 2015-2026. ChipWhisperer is a trademark of NewAE Technology Inc., claimed in all jurisdictions, and registered in at least the United States of America, European Union, and Peoples Republic of China.

Tutorials derived from our open-source work must be released under the associated open-source license, and notice of the source must be *clearly displayed*. Only original copyright holders may license or authorize other distribution - while NewAE Technology Inc. holds the copyright for many tutorials, the github repository includes community contributions which we cannot license under special terms and **must** be maintained as an open-source release. Please contact us for special permissions (where possible).

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.</small>